# Lesson 03 - Agentic Design Patterns

ဒီသင်ခန်းစာမှာတော့ ထိရောက်တဲ့ AI အေးဂျင့်တွေကို တည်ဆောက်ရာမှာ အခြေခံဖြစ်တဲ့ ဒီဇိုင်းပုံစံသုံးခုကို စူးစမ်းလေ့လာပါမယ်။

1. **Clear Agent Instructions** — အေးဂျင့်ပြုလုပ်ရာမှာ ရည်ရွယ်ချက်ရှင်းလင်းတဲ့ prompt တွေဖန်တီးခြင်း
2. **Structured Output with Pydantic Models** — အေးဂျင့်တွေက ကြိုတင်ခန့်မှန်းနိုင်ပြီး အတည်ပြုထားသော ဒေတာများ ပြန်ပေးရန် certainချက်ထားခြင်း
3. **Single Responsibility Agents** — တစ်ခုချင်းစီတာဝန်ရှိပြီး အထူးပြု အေးဂျင့်များ ဒီဇိုင်းဖန်တီးခြင်း

ဒီပုံစံတိုင်းကို **စီးပွားရေးခရီးစဉ်တည်နေရာ အကြံပြုသူ** ကိစ္စတွင် လိုက်နာပြီး အဆင့်တိုင်းမှာ ခရီးစဉ်နေရာများ ထုတ်ပြန်ခြင်း၊ ရနိုင်မှု စစ်ဆေးခြင်း၊ နယ်ပယ်ဆိုင်ရာ စီမံခန့်ခွဲမှုကို တဖြည်းဖြည်း တည်ဆောက်သွားပါမယ်။


## တပ်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ပုံစံ ၁: ပေးပို့သူအတွက် ရှင်းလင်းသော်လည်း အသေးစိတ်ညွှန်ကြားချက်များ

အကျိုးသက်ရောက်မှုအများဆုံး ပုံစံမှာလည်း လွယ်ကူဆုံးဖြစ်ပြီး သင့်ပေးပို့သူအတွက် ရှင်းလင်းသော၊ အသေးစိတ်ညွှန်ကြားချက်များရေးသားခြင်းဖြစ်သည်။

ကောင်းမွန်သောညွှန်ကြားချက်များသည် အောက်ပါအချက်များကို သတ်မှတ်ပေးသည်-
- **ဘယ်သူလဲ** (ပုဂ္ဂိုလ်သဘောနှင့် အသံဖမ်း)
- **ဘာကို** လုပ်သင့်သည် (အဆင့်ပေါင်းလမ်းညွှန်ရေးဆွဲချက်များ)
- **ဘယ်လို** စမ်းသပ်ရေးလုပ်ဆောင်သင့်သည် (ကန့်သတ်ချက်များနှင့် စိန်လျှောက်နည်း)

အောက်တွင် ကျွန်ုပ်တို့သည် တစ်ဦးတည်းသော အဖြေစာလုံးတိုကို ဖန်တီးပေးသော ခရီးသွားဝန်ဆောင်မှုများ ပေးပို့သူကို ရေးဆွဲထားပါသည်။


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

Free-form text သည် စကားပြောရန် အထူးအသုံးဝင်သော်လည်း၊ အောက်ဆုံး စနစ်များအတွက် သတ်မှတ်ထားသော ဒေတာလိုအပ်သည်။
**Pydantic models** နှင့် **tool function** ကိုတွဲဖက် အသုံးပြုခြင်းဖြင့် -

- Agent ၏ output အတွက် တိကျသော schema ကို သတ်မှတ်နိုင်သည်
- အလိုအလျောက်တုံ့ပြန်မှုများကို စစ်ဆေးနိုင်သည်
- Agent  ရလဒ်များကို အပ်လီကေးရှင်း logic ထဲသို့ ယုံကြည်စိတ်ချစွာ ပေါင်းစပ်နိုင်သည်

အထက်ပါအရာများနှင့်အတူ agent အကြံပြုချက်များကို တည်မြဲစေဖို့ destination အသေးစိတ်ကို ပြန်ပေးသော tool တစ်ခုကိုလည်း မိတ်ဆက်ပေးသည်။


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

ခက်ခဲရှုပ်ထွေးသော လုပ်ငန်းများသည် တစ်ဦးချင်းတာဝန်ယူသော agent များစွာကို ခွဲခြားတာဝန်ပေးခြင်းဖြင့် အကျိုးရှိသည်-

- နေရာများနှင့် ရရှိနိုင်မှုအကြောင်း သိရှိသော **Destination Expert**
- လေယာဉ်ခရီးစဉ်များ၊ ဟိုတယ်နှင့် ခရီးစဉ်များကို ကိုင်တွယ်စီမံသူ **Logistics Planner**

ဒါဟာ ဆော့ဖ်ဝဲအင်ဂျင်နီယာနည်းပညာမှ *စိတ်ပူပန်မှု ခွဲခြားမှု* စည်းကမ်းချက်အား ကူးယူထားခြင်းဖြစ်ပြီး agent တစ်ဦးချင်းစီကို စမ်းသပ်ရန်၊ ပြုပြင်ထိန်းသိမ်းရန်နှင့် တိုးတက်အောင်လုပ်ရန် ပိုမိုလွယ်ကူစေသည်။


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## စာရေးရာအကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် ခရီးသွား အကြံပြုသူကဏ္ဍအတွက် အေးဂျင့်ဒီဇိုင်းပုံစံသုံးခုကို သုံးခဲ့ပါသည်-

| ပုံစံ | အဓိကအကြောင်းအရာ | အကျိုးကျေးဇူး |
|---|---|---|
| **ရှင်းလင်းသော ညွှန်ကြားချက်များ** | ပုဂ္ဂိုလ်ရေး၊ တာဝန်များနှင့် ကန့်သတ်ချက်များကို အရင်တည်းက သတ်မှတ်ပေးခြင်း | တူညီသော၊ အမှတ်တံဆိပ်ကို ကိုက်ညီသော အေးဂျင့် လုပ်ဆောင်ချက် |
| **ဖွဲ့စည်းထားသော ထွက်ရှိမှု** | တုံ့ပြန်ချက်ဖော်မတ်အဖြစ် Pydantic မော်ဒယ်များကို အသုံးပြုခြင်း | စစ်ဆေးပြီး၊ စက်အဖတ်နိုင်သည့် ရလဒ်များ |
| **တစ်ခုတည်းသော တာဝန်** | အေးဂျင့်တိုင်းကို တစ်ခုတည်းသော အာရုံစိုက်ထားသော အလုပ်ပေးခြင်း | စမ်းသပ်ရန်၊ ထိန်းသိမ်းရန် နှင့် ပေါင်းစပ်ဖွဲ့စည်းရန် ပိုမိုလွယ်ကူခြင်း |

ဤပုံစံများသည် သဘာဝအတိုင်း ပေါင်းစပ်နိုင်သည် - သင့်တော်သော အေးဂျင့်တစ်ဦးအတွင်း ရှင်းလင်းသောညွှန်ကြားချက်များနှင့် ဖွဲ့စည်းထားသော ထွက်ရှိမှုကို ပေါင်းစပ်၍ ခိုင်မာပြီး ထုတ်လုပ်မှုအဆင်သင့် စနစ်များကို ဆောက်လုပ်နိုင်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ဤစာသားသတိပြုရန်**  
ဤစာတမ်းကို AI ဘာသာပြန်ခြင်းဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြုပြီး ဘာသာပြန်ထားသည်။ တိကျမှုအတွက် ကြိုးပမ်းဆောင်ရွက်သောကြောင့်ဖြစ်ပါသော်လည်း၊ စက်ရုပ်ဘာသာပြန်မှုသည် အမှားများ သို့မဟုတ် မှားယွင်းချက်များပါဝင်နိုင်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ထောက်ခံရရှိသင့်သောအရင်းအမြစ်အနေဖြင့် ယူဆသင့်ပါသည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက်တော့ လူ့ဘာသာပြန်ကြီးများမှ ဘာသာပြန်ပေးခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်မှ ဖြစ်ပေါ်နိုင်သည့် နားမလည်ခြင်း သို့မဟုတ် မမှန်ကန်မှုများအတွက် မည်သည့် တာဝန်မျှမရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
